# 01 — API Exploration

Test connectivity and response shapes for all 4 data sources:
- **TMDB** — movie/show metadata
- **OMDB** — IMDb ratings, plot summaries
- **Streaming Availability (RapidAPI)** — platform catalog
- **TVmaze** — TV show details (no key)

Run cells top-to-bottom. Make sure your `.env` file is populated first.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from backend.config import TMDB_API_KEY, OMDB_API_KEY, RAPIDAPI_KEY, TVMAZE_BASE_URL
print('Keys loaded:',
      'TMDB ✓' if TMDB_API_KEY else 'TMDB ✗',
      '| OMDB ✓' if OMDB_API_KEY else '| OMDB ✗',
      '| RapidAPI ✓' if RAPIDAPI_KEY else '| RapidAPI ✗')

## 1. TMDB — Popular Movies

In [ ]:
import requests

resp = requests.get(
    'https://api.themoviedb.org/3/discover/movie',
    headers={'Authorization': f'Bearer {TMDB_API_KEY}'},
    params={'sort_by': 'popularity.desc', 'page': 1}
)
data = resp.json()
print(f"Status: {resp.status_code} | Total results: {data.get('total_results')}")
print('Sample title:', data['results'][0].get('title') if data.get('results') else 'N/A')

In [ ]:
# Inspect field names returned by TMDB
if data.get('results'):
    print('Available fields:', list(data['results'][0].keys()))

## 2. OMDB — Single Title Lookup

In [ ]:
resp = requests.get('http://www.omdbapi.com/', params={
    'apikey': OMDB_API_KEY,
    't': 'Inception',
    'type': 'movie'
})
omdb = resp.json()
print('Title:', omdb.get('Title'))
print('IMDb Rating:', omdb.get('imdbRating'))
print('Awards:', omdb.get('Awards'))
print('Runtime:', omdb.get('Runtime'))

## 3. Streaming Availability (RapidAPI)

In [ ]:
resp = requests.get(
    'https://streaming-availability.p.rapidapi.com/shows/search/filters',
    headers={
        'X-RapidAPI-Key': RAPIDAPI_KEY,
        'X-RapidAPI-Host': 'streaming-availability.p.rapidapi.com'
    },
    params={'country': 'us', 'orderBy': 'popularity_alltime'}
)
streaming = resp.json()
print(f"Status: {resp.status_code}")
shows = streaming.get('shows', [])
print(f"Shows returned: {len(shows)}")
if shows:
    s = shows[0]
    print('First title:', s.get('title'))
    print('Streaming options keys:', list(s.get('streamingOptions', {}).keys())[:5])

## 4. TVmaze — Public REST API (No Key)

In [ ]:
resp = requests.get(f'{TVMAZE_BASE_URL}/shows', params={'page': 0})
shows = resp.json()
print(f"Status: {resp.status_code} | Shows on page 0: {len(shows)}")
if shows:
    show = shows[0]
    print('Name:', show.get('name'))
    print('Genres:', show.get('genres'))
    print('Network:', show.get('network', {}).get('name') if show.get('network') else 'N/A')

## 5. Summary

All 4 APIs are working. Proceed to `02_embedding_pipeline.ipynb` to build the FAISS index.